In [ ]:
import pyreadstat
import pandas as pd

_, meta = pyreadstat.read_sav("../data/GSS2024.sav", metadataonly=True)

print(meta.file_label)  # often includes the release name
print(meta.number_rows, meta.number_columns)
print(meta.notes)  # sometimes has release info

None
3986 989
[]


In [7]:
cols = set(meta.column_names)
for v in ["occ10", "satjob", "wtssps", "wrkstat"]:
    print(v, v in cols, "->", meta.column_names_to_labels.get(v))

occ10 True -> R's census occupation code (2010)
satjob True -> Work satisfaction
wtssps True -> Person post-stratification weight
wrkstat True -> Labor force status


In [ ]:
"""pyreadstat turns GSS's special codes ("don't know," "inapplicable," "no answer") into NaN"""

keywords = ["important", "job", "work", "prefer"]
for name, label in meta.column_names_to_labels.items():
    if label and any(k in label.lower() for k in keywords):
        print(name, "|", label)

affrmact | Favor preference in hiring Black people
aiworry | R worried that AI will take over many jobs done by humans
amancstr | How important to have American ancestry
ambornin | How important to have been born in America
amchrstn | How important to be a christian
amcit | How important to have American citizenship
amfeel | How important to feel American
amgovt | How important to respect America's laws etc
amtv | TV should give preference to American films
coevwork | Partner ever work as long as a year
cohrs1 | Number of hours partner worked last week
cohrs2 | No. of hours partner usually works a week
corel | Partner's religious preference
cowork1 | Person #1 is R's coworker
cowork2 | Person #2 is R's coworker
cowork3 | Person #3 is R's coworker
cowork4 | Person #4 is R's coworker
cowork5 | Person #5 is R's coworker
cowrkslf | Partner self-emp. or works for somebody else
cowrkslf2 | PARTNER SELF-EMPLOYED, WITH EMPLOYEES, OR WORK FOR SOMEONE ELSE
cowrkslffam | PARTNER WORK AT FAMILY BU

In [10]:
keep = ["occ10", "satjob", "wtssps", "wrkstat"]  # + work-values vars from Step 2
df, meta = pyreadstat.read_sav("../data/GSS2024.sav", usecols=keep)
print(df.shape)

(3986, 4)


In [11]:
print(meta.variable_value_labels.get("satjob"))
print(df["satjob"].value_counts(dropna=False))

{-100.0: 'iap', -99.0: 'no answer', -98.0: "don't know", -97.0: 'skipped on web', -96.0: 'see codebook', -95.0: 'uncodeable', -94.0: 'not imputable', -90.0: 'refused', -80.0: 'not available in this release', -70.0: 'not available in this year', -60.0: "I don't have a job", -40.0: 'dk, na, iap', 1.0: 'very satisfied', 2.0: 'moderately satisfied', 3.0: 'a little dissatisfied', 4.0: 'very dissatisfied'}
satjob
NaN    1204
2.0    1188
1.0    1162
3.0     294
4.0     138
Name: count, dtype: int64


In [17]:
occ = (
    df.dropna(subset=["occ10"])
    .groupby("occ10")["wtssps"]
    .sum()
    .sort_values(ascending=False)
)
occ_pct = (occ / occ.sum() * 100).round(2)

unweighted = df["occ10"].value_counts()
summary = pd.DataFrame({"weighted_pct": occ_pct, "raw_n": unweighted})
summary = summary.sort_values("weighted_pct", ascending=False)
print(summary.head(30))
print("occupations with raw_n < 10:", (summary["raw_n"] < 10).sum())


        weighted_pct  raw_n
occ10                      
430.0           2.94     96
9130.0          2.74     74
4760.0          2.41     76
2310.0          2.40     94
4700.0          2.25     59
3600.0          1.91     87
5700.0          1.68     75
5240.0          1.54     77
3255.0          1.48     64
4720.0          1.48     64
4110.0          1.42     46
6260.0          1.37     43
9620.0          1.33     61
4220.0          1.31     54
4230.0          1.23     49
1020.0          1.14     47
800.0           1.11     51
8965.0          1.10     42
2320.0          1.10     37
5620.0          1.07     46
2000.0          1.00     27
4610.0          0.96     31
2100.0          0.93     27
2540.0          0.93     34
5860.0          0.92     46
1760.0          0.90     13
3930.0          0.86     29
4020.0          0.85     41
4600.0          0.84     30
2200.0          0.84     37
occupations with raw_n < 10: 274


In [15]:
needed = ["occ10", "satjob", "wtssps"]  # + work-values vars
complete = df.dropna(subset=needed)
print("usable rows:", len(complete), "of", len(df))

usable rows: 2144 of 3986


In [16]:
more = ["income", "secur", "promot", "interest", "independ", "help", "value", "hours", "meaning"]
for name, label in meta.column_names_to_labels.items():
    if label and any(k in label.lower() for k in more):
        print(name, "|", label)

abhelp1 | R would help with arrangements for abortion
abhelp2 | R would help with paying for abortion
abhelp3 | R would help with paying for abortion-related other costs
abhelp4 | R would help with emotional support for abortion
abpoor | Low income--cant afford more children
abpoorg | Low income--cant afford more children (grid on web)
amownway | America should follow its own interests
cohrs1 | Number of hours partner worked last week
cohrs2 | No. of hours partner usually works a week
coninc | Family income in constant dollars
conrinc | Respondent income in constant dollars
discaffm | A man won't get a job or promotion
discaffw | A woman won't get a job or promotion
discaffwnv | A woman won't get a job or promotion (no volunteered response on web)
discaffwv | A woman won't get a job or promotion (with volunteered response on web)
eqwlth | Should govt reduce income differences
fehire | Should hire and promote women
finrela | Opinion of family income
helpful | People helpful or looking o

In [18]:
gap = df[df["satjob"].notna() & df["occ10"].isna()]
print(gap["wrkstat"].value_counts())

wrkstat
1.0    58
7.0    48
4.0    18
2.0    11
3.0     7
Name: count, dtype: int64
